# Basic Visualization of an Atmospheric Sounding Profile from UWyoming Sounding Hub
************************************************************************
This script will plot a basic SkewT profile. Retrieve sounding data here. [UWyoming](https://weather.uwyo.edu/upperair/sounding.shtml)

In [1]:
# import necessary packages and modules
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import metpy.calc as mpcalc
from metpy.plots import SkewT, Hodograph
from metpy.units import units
from datetime import datetime
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

# open data and format properly
path = '/home/jovyan/Desktop/Data/2024030506-01028.csv' 
file = pd.read_csv(f'{path}')
file.reset_index(drop=True, inplace=True)
file

,time,longitude,latitude,pressure_hPa,geopotential height_m,temperature_C,dew point temperature_C,ice point temperature_C,relative humidity_%,humidity wrt ice_%,mixing ratio_g/kg,wind direction_degree,wind speed_m/s
0,2024-03-05 07:11:45,19.0019,74.5036,1025.5,20,0.2,-2.1,-1.9,85,85,3.19,237,3.2
1,2024-03-05 07:11:47,19.0021,74.5036,1024.3,30,0.5,-2.0,-1.8,83,83,3.21,241,1.5
2,2024-03-05 07:11:49,19.0023,74.5037,1022.7,42,0.7,-2.3,-2.0,80,80,3.16,240,1.6
3,2024-03-05 07:11:51,19.0025,74.5037,1021.4,52,0.7,-2.6,-2.3,79,79,3.09,239,2.0
4,2024-03-05 07:11:53,19.0029,74.5037,1020.2,62,0.7,-2.7,-2.4,78,78,3.08,238,2.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999,2024-03-05 08:51:13,18.8057,74.3581,11.8,29006,-43.0,-80.6,-75.8,1,1,0.06,82,52.0
3000,2024-03-05 08:51:15,18.8024,74.3580,11.7,29019,-42.8,-80.5,-75.7,1,1,0.06,82,52.2
3001,2024-03-05 08:51:17,18.7990,74.3579,11.7,29033,-42.7,-80.4,-75.6,1,1,0.06,82,52.3
3002,2024-03-05 08:51:19,18.7956,74.3578,11.7,29048,-42.6,-80.3,-75.6,1,1,0.06,82,52.3


In [2]:
# get date
date_og = path[26:34]
date_object = datetime.strptime(date_og, "%Y%m%d")
date = date_object.strftime("%Y-%m-%d")
print(date)

# change values to floats
def change_float(variable):
    n_var = np.array(variable, dtype = float) 
    return(n_var)

2024-03-05


In [3]:
# get values
time = file['time'].values
pressure = file['pressure_hPa'].values*units.hPa
temp = file['temperature_C'].values*units.degC
td = file['dew point temperature_C'].values*units.degC
rh = file['relative humidity_%'].values*units('%')
speed = file['wind speed_m/s'].values*1.94384 # convert to kts
dir = file['wind direction_degree'].values
geopotent = file['geopotential height_m'].values*(1/1000) # convert to km

# convert string format to float format
speed = change_float(speed)
dir = change_float(dir)
td = change_float(td)
rh = change_float(rh)
geopotent = change_float(geopotent)
pressure = change_float(pressure)
temp = change_float(temp)

# apply units
pressure = pressure * units.hPa  
temp = temp * units.degC         
td = td * units.degC    
rh = rh * units('%')
speed = speed * units.kts
dir = dir * units.degree
geopotent = geopotent * units.km

# compute u and v components of wind speed and direction
u, v = mpcalc.wind_components(speed, dir)
print('All variables have been established. Now beginning calculations...')

All variables have been established. Now beginning calculations...


In [ ]:
# specify if you want a trimmed view or not
specs = {
    'full': (10, 15, False, -50, 20),
    'trimmed': (10, 5, True, -38, 8)
}
var = 'full'
info = specs.get(var, ('k', 'invalid')) # black cmap and 'invalid' name is the default if var isn't found
minsize, maxsize, ylim, xlimmin, xlimmax = info[0], info[1], info[2], info[3], info[4]

# plot data
fig = plt.figure(figsize=(minsize,maxsize), facecolor = 'black') 

# SKEW-T PLOT
skew_ax = fig.add_subplot(projection='skewx')
skew = SkewT(fig, rotation = 45, subplot = skew_ax)
skew.ax.set_facecolor(color = 'black')

if ylim == True:
    skew.ax.set_ylim(1000,500) 

skew.ax.tick_params(axis='y', colors='white', labelsize = 15) # change y axis hPa standard levels from black to white color
skew.ax.set_xlim(xlimmin, xlimmax) 
skew.ax.tick_params(axis='x', colors='white', labelsize = 15) # change x axis C standard values from black to white color

# plot temp, td, and parcel profile lines
skew.plot(pressure[0:len(pressure)], temp[0:len(temp)], 'red')
skew.plot(pressure[0:len(pressure)], td[0:len(td)], 'lime')

# plot wind barbs
barb_num = 35
skew.plot_barbs(pressure[::barb_num], u[::barb_num], v[::barb_num], xloc = 1.0, x_clip_radius=0.1, y_clip_radius=0.08, color = 'white') # set [::#] for better reading - less packed

# plot other important lines
skew.plot_dry_adiabats(linestyle = '--')
skew.plot_moist_adiabats(linestyle = '--')
skew.plot_mixing_lines(linestyle = '--')

plt.title(f'Sounding from Bear Island (BJORNOYA), Norway \nValid 06 UTC {date}', fontsize=17, color='white', fontweight='bold')
## OPTIONAL: Add a legend ##
# # set legend info using mpatches - going to have 6 columns, 3 each
# temp_patch = Line2D([], [], color='tab:red', marker='o', linestyle='None', markersize=3, label='Temp.')
# dewpt_patch = Line2D([], [], color='tab:green', marker='o', linestyle='None', markersize=3, label='Dewpt.')

# # plot the legend itself under the plot
# legend = plt.legend(handles=[temp_patch, dewpt_patch], labelcolor='white',loc='upper center', bbox_to_anchor=(0.5, -0.08),
#                     fancybox=True, shadow=True, ncol=4, facecolor = 'black', 
#                     edgecolor = 'white', framealpha = 0.8, fontsize = 12) 

plt.tight_layout()
plt.savefig(f'/home/jovyan/Desktop/Figures/{var}_profile_{date}_06UTC.png',dpi=300)